## Подготовка данных

In [33]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

ROOT = Path.cwd().resolve().parent

CSV_PATH = ROOT / "data" / "training" / "attendance_synthetic.csv"
MODEL_PATH = ROOT / "models" / "xgboost_attendance.ubj"

In [34]:
if not CSV_PATH.is_file():
    raise FileNotFoundError(
        f"Положите датасет рядом с проектом. Ожидался файл: {CSV_PATH}"
    )

df = pd.read_csv(CSV_PATH)
print("Loaded", len(df), "rows")
print(df.head())

Loaded 400 rows
     discipline_name tournament_type  hour  day_of_week  month  prize_pool  \
0  Rainbow Six Siege         offline    13            6      3       54000   
1           Fortnite          online    16            2      3       62000   
2           Fortnite         offline    23            1      2       18200   
3  League of Legends         offline    16            1      1      120000   
4  League of Legends          online    14            5      6      169000   

   registered_count  actual_attendance  
0               402                329  
1               463                  4  
2               282                207  
3              1226               1047  
4              1060                 13  


In [35]:
TARGET = 'actual_attendance'
FEATURES = df.columns.drop(TARGET).tolist()

cat_features = [c for c in ['discipline_name', 'tournament_type'] if c in df.columns]
num_features = [c for c in FEATURES if c not in cat_features]

X = df[FEATURES].copy()
y = df[TARGET].copy().astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20
)

for col in cat_features:
    if col in X_train.columns:
        X_train[col] = X_train[col].astype('category')
        X_test[col] = X_test[col].astype('category')

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

Train shape: (320, 7), Test shape: (80, 7)


## XGBoost

In [36]:
import optuna
from xgboost import XGBRegressor


def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 600, 1500),
        'max_depth': trial.suggest_int('max_depth', 6, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0.0, 0.5),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 2.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 2.0),
    }

    model = XGBRegressor(
        **params,
        objective='reg:squarederror',
        eval_metric='mae',
        enable_categorical=True,
        tree_method='hist',
        early_stopping_rounds=80
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=False
    )

    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    return mae

study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=40)

print(f"Лучшее MAE: {study.best_value:.3f}")
print("Лучшие параметры:")
for key, value in study.best_params.items():
    print(f"  {key:18}: {value}")

[I 2026-03-30 21:07:38,534] A new study created in memory with name: no-name-e7fc2917-13e6-4634-978b-8928390c4ce3
[I 2026-03-30 21:07:39,003] Trial 0 finished with value: 84.59134674072266 and parameters: {'n_estimators': 937, 'max_depth': 12, 'learning_rate': 0.07259248719561363, 'subsample': 0.8795975452591109, 'colsample_bytree': 0.7468055921327309, 'min_child_weight': 2, 'gamma': 0.02904180608409973, 'reg_lambda': 1.7323522915498704, 'reg_alpha': 1.2022300234864176}. Best is trial 0 with value: 84.59134674072266.
[I 2026-03-30 21:07:39,217] Trial 1 finished with value: 83.51225280761719 and parameters: {'n_estimators': 1237, 'max_depth': 6, 'learning_rate': 0.13826189316223852, 'subsample': 0.9497327922401265, 'colsample_bytree': 0.7637017332034828, 'min_child_weight': 2, 'gamma': 0.09170225492671691, 'reg_lambda': 0.6084844859190754, 'reg_alpha': 1.0495128632644757}. Best is trial 1 with value: 83.51225280761719.
[I 2026-03-30 21:07:39,334] Trial 2 finished with value: 102.2251052

Лучшее MAE: 40.825
Лучшие параметры:
  n_estimators      : 639
  max_depth         : 8
  learning_rate     : 0.010130470338732663
  subsample         : 0.7812699927417589
  colsample_bytree  : 0.9280142336783304
  min_child_weight  : 2
  gamma             : 0.05328191267652074
  reg_lambda        : 0.49213656270441686
  reg_alpha         : 0.31310007980049837


In [37]:
best_params = study.best_params
final_model = XGBRegressor(
    **best_params,
    objective='reg:squarederror',
    eval_metric='mae',
    enable_categorical=True,
    tree_method='hist',
    random_state=42,
    early_stopping_rounds=100
)

final_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.9280142336783304
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",100
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegress

In [38]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

final_model.save_model(str(MODEL_PATH))
print(f"\nModel saved to: {MODEL_PATH}")


Model saved to: D:\Projects\Coursework\CyberTracker\backend\models\xgboost_attendance.ubj


In [39]:
y_pred = final_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:.1f}")
print(f"RMSE: {rmse:.1f}")
print(f"R²:   {r2:.4f}")

MAE: 42.5
RMSE: 79.2
R²:   0.9882
